# Identifier une molécule à partir de ses spectres de masse (CASMI 2026)

## 1. Problème et démarche

Pour chaque molécule inconnue, on dispose d'un ou plusieurs spectres MS/MS et on propose jusqu'à 25 SMILES classés. Le score est le MRR@25 : la moyenne de 1/rang de la bonne structure, 0 si elle n'est pas dans les 25. Une structure est juste si les 14 premiers caractères de son InChIKey sont les bons ; la stéréochimie ne compte pas.

Un classifieur à une classe par molécule ne reconnaîtrait que des molécules déjà vues. On prédit donc un fingerprint de Morgan, puis on cherche les structures proches dans un catalogue :

1. masse neutre, à partir de `precursor_mz` et de l'`adduct` ;
2. candidats du catalogue à ±10 ppm ;
3. fingerprint prédit à partir du spectre (seule étape apprise) ;
4. tri des candidats par similarité de Tanimoto, et top 25.

La validation porte sur des molécules absentes de l'entraînement. Le test visible n'y sert pas, car ses spectres sont des copies de spectres du train.

### Imports et chemins

Les données sont lues dans `data/raw/` en local et dans `/kaggle/input/` sur Kaggle. Les fichiers produits vont dans `data/rapport/`, ce qui évite d'écraser ceux de l'ancien code. La cellule doit afficher les trois fichiers.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

SEED = 42

KAGGLE_INPUT = Path("/kaggle/input")
if KAGGLE_INPUT.exists():
    RAW_DIR = next(KAGGLE_INPUT.rglob("train.parquet")).parent
    WORK_DIR = Path("/kaggle/working")
else:
    RAW_DIR = Path("data/raw")
    WORK_DIR = Path("data/rapport")
WORK_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = RAW_DIR / "train.parquet"
TEST_PATH = RAW_DIR / "test.parquet"
SUBMISSION_PATH = RAW_DIR / "sample_submission.csv"

for path in [TRAIN_PATH, TEST_PATH, SUBMISSION_PATH]:
    print(f"{path}  {path.stat().st_size / 1e6:,.1f} Mo")

## 2. Données

On charge seulement les colonnes utiles du train, sans les pics, qui sont les plus lourds. L'index de `train` est la position du spectre dans le fichier : il servira à relire les pics d'un échantillon. À regarder : le nombre de spectres et de molécules.

Constat : le train compte 2 539 608 spectres pour 275 810 molécules et tient en 0,7 Go sans les pics. Le test visible compte 1 213 spectres pour 400 molécules.

In [ ]:
COLUMNS = ["inchikey14", "normalized_smiles", "ingest_lib", "adduct", "ionization_mode",
           "instrument_type", "precursor_mz", "collision_energy_ev", "num_peaks"]

train = pd.read_parquet(TRAIN_PATH, columns=COLUMNS)
test = pd.read_parquet(TEST_PATH)

print(f"train : {len(train):,} spectres, {train['inchikey14'].nunique():,} molécules, "
      f"{train.memory_usage(deep=True).sum() / 1e9:.1f} Go en mémoire")
print(f"test  : {len(test):,} spectres, {test['molecule_id'].nunique():,} molécules")
train.head(3)

### Provenance des spectres

Le train réunit plusieurs bibliothèques de spectres (`ingest_lib`), mesurées dans des laboratoires et sur des appareils différents. Le graphique compte les spectres de chacune, et le tableau ajoute leur part et leur nombre de molécules. À regarder : le poids de la première bibliothèque par rapport aux autres.

In [ ]:
counts = train["ingest_lib"].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(counts.index, counts.values / 1e3, color="#2a78d6", height=0.7)
ax.set_xlabel("spectres (milliers)")
ax.set_title("Spectres du train par bibliothèque", loc="left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

molecules = train.groupby("ingest_lib")["inchikey14"].nunique()
pd.DataFrame({"spectres": counts, "% des spectres": (counts / len(train) * 100).round(1),
              "molécules": molecules}).sort_values("spectres", ascending=False)